# Exp5 map positions - reusable several-fish analysis

Short orchestration notebook derived from `Exp5_map_positions_several.ipynb`. Reusable analysis, summary, and plotting helpers live in `src/`; this notebook keeps only experiment choices and display cells.


In [ ]:
# Cell 01 - Setup imports
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import importlib

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import src.data_loading as exio
import src.analysis_tools as at
import src.multifish_analysis as mfa
import src.plotting as plott

importlib.reload(exio)
importlib.reload(at)
importlib.reload(mfa)
importlib.reload(plott)


In [ ]:
# Cell 02 - Experiment config
experiment_name = "Exp_5_map_positions"

def first_existing_path(*candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    tried = "\n".join(str(Path(candidate)) for candidate in candidates)
    raise FileNotFoundError("None of the configured root paths exists:\n" + tried)

onedrive_base = Path(r"D:\Alejandro\Data")
onedrive_candidates = [
    onedrive_base / "OneDrive - Universite de Lausanne",
    *sorted(onedrive_base.glob("OneDrive - Universit* de Lausanne")),
]
onedrive_root = first_existing_path(*onedrive_candidates)
main_path = onedrive_root / "Lab" / "Data" / "2p"
analysis_path = onedrive_root / "Lab" / "Analysis"
stimuli_main_path = analysis_path
save_path = analysis_path / experiment_name / "plots"

fps_2p = 2.0
selected_blocks = [f"B{n}" for n in range(1, 5)]
t_pre_s = 5.0
t_post_s = 25.0
motion_onset_s = 8.0
tau_s = 6.0
motion_duration_key = "motion_sec"

fish_ids = [
    "L684_f01",
    "L684_f07",
    "L733_f01",
    "L733_f06",
    "L733_f07",
]

analysis_label = "Left"
selected_stimuli_to_plot = ["LeB1", "LeB2", "LeB3", "LeB4"]  # use None for all detected stimuli
left_right_stimuli = ("LeBcontrol", "RiBcontrol")
left_right_filter_mode = "left"  # "none", "abs", "left", "right", or "range"
left_right_threshold = 0.3
left_right_range = (-0.3, 0.3)

print("Using main_path:", main_path)
print("Using analysis_path:", analysis_path)


In [ ]:
# Cell 03 - Load and align all fish
all_fish_data = {}
for fish_id in fish_ids:
    print(f"Loading {fish_id}")
    all_fish_data[fish_id] = exio.load_and_align_2p_experiment(
        fish_id=fish_id,
        experiment_name=experiment_name,
        main_path=main_path,
        stimuli_main_path=stimuli_main_path,
        fps_2p=fps_2p,
        selected_blocks=selected_blocks,
        t_pre_s=t_pre_s,
        t_post_s=t_post_s,
        verbose=False,
    )

reference_fish_id = fish_ids[0]
reference_fish = all_fish_data[reference_fish_id]
print(f"Loaded {len(all_fish_data)} fish")
print("Detected stimuli:", reference_fish["stimuli_names"])


In [ ]:
# Cell 04 - Stimulus detection, styles, and plot selection
response_stimuli = list(reference_fish["stimuli_names"])
response_selection = at.resolve_selected_stimuli(
    response_stimuli,
    stimuli_id_map=reference_fish["stimuli_id_map"],
    available_stimuli=reference_fish["trial_aligned_traces_z_core"].keys(),
)
response_stimulus_ids = response_selection["stimulus_ids"]
response_stimulus_labels = response_selection["stimulus_labels"]

if selected_stimuli_to_plot is None:
    plot_selection_input = response_stimulus_labels
else:
    plot_selection_input = selected_stimuli_to_plot
plot_selection = at.resolve_selected_stimuli(
    plot_selection_input,
    stimuli_id_map=reference_fish["stimuli_id_map"],
    available_stimuli=reference_fish["trial_aligned_traces_z_core"].keys(),
)
plot_stimulus_ids = plot_selection["stimulus_ids"]
plot_stimulus_labels = plot_selection["stimulus_labels"]

stimuli_colors, stimuli_linestyles = plott.build_stimulus_style_maps(
    stimuli_names=response_stimulus_labels,
)
stimuli_ordered = response_stimulus_labels
stimuli_durations = reference_fish["stimuli_durations"]

print("All response stimuli:", response_stimulus_labels)
print("Plot stimuli:", plot_stimulus_labels)


In [ ]:
# Cell 05 - Response matrices and response-window validation
response_window_rows = []
for fid in fish_ids:
    fish = all_fish_data[fid]
    for stim_id, stim_label in zip(response_stimulus_ids, response_stimulus_labels):
        key = stim_id if stim_id in fish["trial_aligned_traces_z_core"] else str(stim_id)
        arr = fish["trial_aligned_traces_z_core"][key]
        window = at.compute_response_window_frames(
            n_time=arr.shape[1],
            fps_2p=fps_2p,
            t_pre_s=t_pre_s,
            motion_onset_s=motion_onset_s,
            stimulus=stim_id,
            stimuli_durations=fish["stimuli_durations"],
            stimuli_id_map=fish["stimuli_id_map"],
            tau_s=tau_s,
            motion_duration_key=motion_duration_key,
        )
        response_window_rows.append({
            "fish_id": fid,
            "stimulus": stim_label,
            "stimulus_id": stim_id,
            "n_time": arr.shape[1],
            "start_frame": window["start_frame"],
            "stop_frame": window["stop_frame"],
            "n_response_frames": window["n_frames"],
            "start_s": window["start_s"],
            "end_s": window["end_s"],
        })
response_window_validation = pd.DataFrame(response_window_rows)

response_results = mfa.build_zscore_response_matrices_all_fish(
    all_fish_data=all_fish_data,
    fish_ids=fish_ids,
    selected_stimuli=response_stimulus_labels,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    motion_onset_s=motion_onset_s,
    tau_s=tau_s,
    motion_duration_key=motion_duration_key,
)
response_matrices_by_fish = response_results["response_matrices"]
pooled_response_matrix = response_results["pooled_response_matrix"]
response_row_metadata = response_results["row_metadata"]

shape_summary = pd.DataFrame([
    {
        "fish_id": fid,
        "response_rows": response_matrices_by_fish[fid].shape[0],
        "kept_neurons": len(all_fish_data[fid]["kept_neuron_indices"]),
        "response_columns": response_matrices_by_fish[fid].shape[1],
    }
    for fid in fish_ids
])
print("Pooled response matrix:", pooled_response_matrix.shape)
display(shape_summary)
display(response_window_validation.groupby("stimulus")["n_response_frames"].describe())


In [ ]:
# Cell 06 - Consolidated per-neuron summary table
active_matrices = mfa.build_active_neuron_matrices_all_fish(
    all_fish_data=all_fish_data,
    fish_ids=fish_ids,
    stim_order=response_stimulus_ids,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    tau_s=tau_s,
    active_fraction_threshold=0.30,
    min_epoch_s=2.0,
    min_active_reps=2,
    expected_reps=4,
)

neuron_summary_all_stimuli_table = mfa.build_neuron_stimulus_summary_table(
    response_matrices=response_matrices_by_fish,
    active_matrices=active_matrices,
    row_metadata=response_row_metadata,
    selected_stimulus_ids=response_stimulus_ids,
    selected_stimulus_labels=response_stimulus_labels,
    analysis_label=analysis_label,
)

neuron_summary_table = mfa.build_neuron_stimulus_summary_table(
    response_matrices=response_matrices_by_fish,
    active_matrices=active_matrices,
    row_metadata=response_row_metadata,
    selected_stimulus_ids=plot_stimulus_ids,
    selected_stimulus_labels=plot_stimulus_labels,
    analysis_label=analysis_label,
)
neuron_summary_table = mfa.add_selectivity_metrics_to_summary_table(
    neuron_summary_table,
    selected_stimulus_labels=plot_stimulus_labels,
)

left_right_index = np.full(pooled_response_matrix.shape[0], np.nan, dtype=float)
left_stimulus, right_stimulus = left_right_stimuli
if left_stimulus in pooled_response_matrix.columns and right_stimulus in pooled_response_matrix.columns:
    auc_left = pooled_response_matrix[left_stimulus].to_numpy(dtype=float)
    auc_right = pooled_response_matrix[right_stimulus].to_numpy(dtype=float)
    denom = auc_left + auc_right
    finite = np.isfinite(denom) & (np.abs(denom) > 1e-12)
    left_right_index[finite] = (auc_left[finite] - auc_right[finite]) / denom[finite]
else:
    print("Left/right index skipped because configured control stimuli were not found.")

if left_right_filter_mode == "none":
    plot_neuron_keep_mask = np.ones(pooled_response_matrix.shape[0], dtype=bool)
elif left_right_filter_mode == "abs":
    plot_neuron_keep_mask = np.isfinite(left_right_index) & (np.abs(left_right_index) >= left_right_threshold)
elif left_right_filter_mode == "left":
    plot_neuron_keep_mask = np.isfinite(left_right_index) & (left_right_index >= left_right_threshold)
elif left_right_filter_mode == "right":
    plot_neuron_keep_mask = np.isfinite(left_right_index) & (left_right_index <= -left_right_threshold)
elif left_right_filter_mode == "range":
    lo, hi = left_right_range
    plot_neuron_keep_mask = np.isfinite(left_right_index) & (left_right_index >= lo) & (left_right_index <= hi)
else:
    raise ValueError("left_right_filter_mode must be 'none', 'abs', 'left', 'right', or 'range'.")

plot_neuron_keep_indices = np.flatnonzero(plot_neuron_keep_mask)
neuron_summary_table["left_right_index"] = left_right_index
neuron_summary_table["plot_neuron_keep"] = plot_neuron_keep_mask
neuron_summary_all_stimuli_table["left_right_index"] = left_right_index
neuron_summary_all_stimuli_table["plot_neuron_keep"] = plot_neuron_keep_mask
selected_auc_response_matrix = pooled_response_matrix.loc[plot_neuron_keep_mask, plot_stimulus_labels].copy()

plot_filter_table = response_row_metadata.copy().reset_index(drop=True)
plot_filter_table["left_right_index"] = left_right_index
plot_filter_table["plot_neuron_keep"] = plot_neuron_keep_mask

print("Neuron summary table:", neuron_summary_table.loc[plot_neuron_keep_mask].shape)
print("All-stimuli response table:", neuron_summary_all_stimuli_table.shape)
print(f"Plot filter kept {plot_neuron_keep_indices.size} of {plot_neuron_keep_mask.size} neurons")
display(plot_filter_table.head())


In [ ]:
# Cell 07 - Stimulus-vector similarity from selected AUC responses
similarity_results = mfa.build_stimulus_vector_similarity(
    selected_auc_response_matrix,
    selected_stimuli=plot_stimulus_labels,
)
pearson_similarity_matrix = similarity_results["pearson_similarity_matrix"]
cosine_similarity_matrix = similarity_results["cosine_similarity_matrix"]
segment_similarity_df = similarity_results["pair_similarity"]

plott.plot_similarity_heatmaps(pearson_similarity_matrix, cosine_similarity_matrix)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
plott.plot_similarity_by_distance(segment_similarity_df, "pearson_similarity", ax=axes[0])
plott.plot_similarity_by_distance(segment_similarity_df, "cosine_similarity", ax=axes[1])
plt.show()

print("Similarity stimuli:", plot_stimulus_labels)
display(segment_similarity_df)


In [ ]:
# Cell 08 - Segment selectivity permutation summary
segments_to_compare = ["B1", "B2", "B3", "B4"]
n_permutations = 1000
random_seed = 42
alpha_percentile = 95

segment_selectivity_results = mfa.build_segment_selectivity_permutation_summary(
    all_fish_data=all_fish_data,
    fish_ids=fish_ids,
    selected_stimulus_ids=plot_stimulus_ids,
    selected_stimulus_labels=plot_stimulus_labels,
    segments_to_compare=segments_to_compare,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    motion_onset_s=motion_onset_s,
    tau_s=tau_s,
    motion_duration_key=motion_duration_key,
    n_permutations=n_permutations,
    random_seed=random_seed,
    alpha_percentile=alpha_percentile,
)
segment_selectivity_summary_df = segment_selectivity_results["summary_df"]
si_real = segment_selectivity_results["si_real"]
si_shuffle = segment_selectivity_results["si_shuffle"]
significant_segment_selective = segment_selectivity_results["significant_segment_selective"]

segment_columns = [
    "global_neuron_id",
    "segment_selectivity_index",
    "preferred_segment",
    "segment_shuffle_threshold",
    "segment_selective",
    "segment_skip_reason",
]
neuron_summary_table = neuron_summary_table.merge(
    segment_selectivity_summary_df[segment_columns],
    on="global_neuron_id",
    how="left",
    validate="one_to_one",
)

print("Segments used:", segment_selectivity_results["segments_requested"])
print("Significant selective neurons:", int(np.count_nonzero(significant_segment_selective)))
print("Skip reasons:", segment_selectivity_results["skip_reason_counts"])
display(segment_selectivity_results["trial_counts"].head())


In [ ]:
# Cell 09 - Segment selectivity plot
finite_shuffle = si_shuffle[np.isfinite(si_shuffle)]
finite_real = si_real[np.isfinite(si_real)]

fig_segment_si, ax_segment_si = plt.subplots(figsize=(7, 4.5))
if finite_shuffle.size > 0:
    ax_segment_si.hist(finite_shuffle, bins=40, color="0.75", alpha=0.75, density=True, label="shuffled SI")
if finite_real.size > 0:
    ax_segment_si.hist(finite_real, bins=25, histtype="step", color="black", linewidth=1.5, density=True, label="real SI")
ax_segment_si.set_xlabel("selectivity index")
ax_segment_si.set_ylabel("density")
ax_segment_si.set_title("Permutation-based segment selectivity")
ax_segment_si.legend(frameon=False, fontsize=8)
fig_segment_si.tight_layout()
plt.show()


In [ ]:
# Cell 10 - Stimulus specificity summary plots
plot_summary_table = neuron_summary_table.loc[neuron_summary_table["plot_neuron_keep"]].reset_index(drop=True)

fig_stimulus_specificity_sparseness, ax_stimulus_specificity_sparseness = plt.subplots(figsize=(6, 4.5))
plott.plot_stimulus_specificity_sparseness(
    plot_summary_table,
    selected_stimulus_labels=plot_stimulus_labels,
    analysis_label=analysis_label,
    ax=ax_stimulus_specificity_sparseness,
)
fig_stimulus_specificity_sparseness.tight_layout()
plt.show()

fig_active_stimuli_histogram, ax_active_stimuli_histogram = plt.subplots(figsize=(6, 4.5))
plott.plot_active_stimuli_histogram(
    plot_summary_table,
    selected_stimulus_labels=plot_stimulus_labels,
    analysis_label=analysis_label,
    ax=ax_active_stimuli_histogram,
)
fig_active_stimuli_histogram.tight_layout()
plt.show()

fig_preferred_stimulus_distribution, ax_preferred_stimulus_distribution = plt.subplots(figsize=(6, 4.5))
plott.plot_preferred_stimulus_distribution(
    plot_summary_table,
    selected_stimulus_labels=plot_stimulus_labels,
    analysis_label=analysis_label,
    ax=ax_preferred_stimulus_distribution,
)
fig_preferred_stimulus_distribution.tight_layout()
plt.show()

fig_stimulus_specificity_selectivity_index, ax_stimulus_specificity_selectivity_index = plt.subplots(figsize=(6, 4.5))
plott.plot_stimulus_specificity_selectivity_index(
    plot_summary_table,
    selected_stimulus_labels=plot_stimulus_labels,
    analysis_label=analysis_label,
    ax=ax_stimulus_specificity_selectivity_index,
)
fig_stimulus_specificity_selectivity_index.tight_layout()
plt.show()


In [ ]:
# Cell 11 - High lifetime-sparseness z-score raster
# Equivalent to the original high-sparseness z-score raster, using the selected plot stimuli.
lifetime_sparseness_threshold = 0.75
high_sparseness_plot_stim_order = plot_stimulus_ids
high_sparseness_preferred_stimulus_order = plot_stimulus_labels
high_sparseness_combine_mode = "mean"
high_sparseness_trace_type = "zscore"

high_sparseness_metric_source_table = neuron_summary_table.copy()
high_sparseness_source_row_count = high_sparseness_metric_source_table.shape[0]

if "plot_neuron_keep_mask" in globals():
    high_sparseness_global_ids = high_sparseness_metric_source_table["global_neuron_id"].to_numpy(dtype=int)
    if plot_neuron_keep_mask.shape[0] <= high_sparseness_global_ids.max(initial=-1):
        raise ValueError(
            "plot_neuron_keep_mask is shorter than the global_neuron_id values in "
            "high_sparseness_metric_source_table."
        )
    high_sparseness_metric_source_table = high_sparseness_metric_source_table.loc[
        plot_neuron_keep_mask[high_sparseness_global_ids]
    ].copy()
    print(
        "Notebook-level neuron filter for Cell 11 kept "
        f"{high_sparseness_metric_source_table.shape[0]} of {high_sparseness_source_row_count} "
        "candidate neurons before lifetime-sparseness filtering."
    )

high_sparseness_response_matrix = mfa.build_matrix_all_fish(
    all_fish_data,
    high_sparseness_plot_stim_order,
    fish_ids=fish_ids,
    combine_mode=high_sparseness_combine_mode,
    trace_type=high_sparseness_trace_type,
)

preference_rank = {
    stimulus: rank
    for rank, stimulus in enumerate(high_sparseness_preferred_stimulus_order)
}
high_sparseness_summary_df = high_sparseness_metric_source_table.loc[
    np.isfinite(high_sparseness_metric_source_table["lifetime_sparseness"])
    & (high_sparseness_metric_source_table["lifetime_sparseness"] > lifetime_sparseness_threshold)
].copy()
high_sparseness_summary_df["_preferred_stimulus_rank"] = (
    high_sparseness_summary_df["preferred_stimulus"]
    .map(preference_rank)
    .fillna(len(preference_rank))
)
high_sparseness_summary_df["_max_auc_sort"] = high_sparseness_summary_df["max_response"].fillna(-np.inf)
high_sparseness_summary_df = high_sparseness_summary_df.sort_values(
    by=["_preferred_stimulus_rank", "_max_auc_sort", "global_neuron_id"],
    ascending=[True, False, True],
    kind="mergesort",
).reset_index(drop=True)

high_sparseness_neuron_order_global = high_sparseness_summary_df["global_neuron_id"].to_numpy(dtype=int)
high_sparseness_zscore_matrix = high_sparseness_response_matrix[high_sparseness_neuron_order_global, :]
high_sparseness_neuron_order = np.arange(high_sparseness_zscore_matrix.shape[0], dtype=int)

print(
    f"Neurons with lifetime sparseness > {lifetime_sparseness_threshold}: "
    f"{high_sparseness_zscore_matrix.shape[0]}"
)
print("Preferred stimulus order:", list(high_sparseness_preferred_stimulus_order))
print("Plot stimulus order:", list(high_sparseness_plot_stim_order))
print("Counts by preferred stimulus:")
print(
    high_sparseness_summary_df["preferred_stimulus"]
    .value_counts()
    .reindex(high_sparseness_preferred_stimulus_order, fill_value=0)
)

display(
    high_sparseness_summary_df[
        [
            "fish_id",
            "neuron_id",
            "source_neuron_id",
            "global_neuron_id",
            "preferred_stimulus",
            "max_response",
            "lifetime_sparseness",
        ]
    ].head()
)

if high_sparseness_zscore_matrix.shape[0] == 0:
    print("No neurons passed the lifetime sparseness threshold; raster was not plotted.")
    fig_high_sparseness_zscore_raster = None
    ax_high_sparseness_zscore_raster = None
    im_high_sparseness_zscore_raster = None
else:
    (
        fig_high_sparseness_zscore_raster,
        ax_high_sparseness_zscore_raster,
        im_high_sparseness_zscore_raster,
        high_sparseness_plot_order,
    ) = plott.plot_allfish_flat_raster(
        data=high_sparseness_zscore_matrix,
        trial_aligned_traces=reference_fish["trial_aligned_traces_z_core"],
        stim_order=high_sparseness_plot_stim_order,
        stimuli_id_map=reference_fish["stimuli_id_map"],
        stimuli_durations=reference_fish["stimuli_durations"],
        stimuli_colors=stimuli_colors,
        stimuli_linestyles=stimuli_linestyles,
        fps_2p=fps_2p,
        t_pre_s=t_pre_s,
        combine_mode=high_sparseness_combine_mode,
        sort_mode="unsorted",
        neuron_order=high_sparseness_neuron_order,
        sort_label=f"sparseness>{lifetime_sparseness_threshold}, preferred stimulus, max AUC",
        is_binary=False,
        show_mean_trace=True,
        figsize=(12, 8),
        fish_id="all_fish_high_lifetime_sparseness_zscore",
        vmax=4,
    )
    plt.show()


In [ ]:
# Cell 12 - Left z-score and active rasters
# Restores the original pooled left-side z-score and binary active rasters.
left_stim_order = [7, 3, 4, 5, 6, 1, 13]
left_combine_mode = "mean"

left_zscore_matrix = mfa.build_matrix_all_fish(
    all_fish_data,
    left_stim_order,
    fish_ids=fish_ids,
    combine_mode=left_combine_mode,
    trace_type="zscore",
)
if "plot_neuron_keep_mask" in globals():
    if plot_neuron_keep_mask.shape[0] != left_zscore_matrix.shape[0]:
        raise ValueError("plot_neuron_keep_mask length does not match z-score matrix rows.")
    left_zscore_matrix = left_zscore_matrix[plot_neuron_keep_mask, :]
print("Final matrix shape:", left_zscore_matrix.shape)

(
    fig_left_zscore_raster,
    ax_left_zscore_raster,
    im_left_zscore_raster,
    left_zscore_neuron_order,
) = plott.plot_allfish_flat_raster(
    data=left_zscore_matrix,
    trial_aligned_traces=reference_fish["trial_aligned_traces_z_core"],
    stim_order=left_stim_order,
    stimuli_id_map=reference_fish["stimuli_id_map"],
    stimuli_durations=reference_fish["stimuli_durations"],
    stimuli_colors=stimuli_colors,
    stimuli_linestyles=stimuli_linestyles,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    combine_mode=left_combine_mode,
    sort_mode="unsorted",
    n_clusters=3,
    random_state=0,
    neuron_order=None,
    sort_label=None,
    is_binary=False,
    show_mean_trace=True,
    figsize=(12, 8),
    fish_id="all_fish",
    vmax=4,
)
plt.show()

left_active_matrix = mfa.build_matrix_all_fish(
    all_fish_data,
    left_stim_order,
    fish_ids=fish_ids,
    combine_mode=left_combine_mode,
    trace_type="raster",
)
if "plot_neuron_keep_mask" in globals():
    if plot_neuron_keep_mask.shape[0] != left_active_matrix.shape[0]:
        raise ValueError("plot_neuron_keep_mask length does not match active matrix rows.")
    left_active_matrix = left_active_matrix[plot_neuron_keep_mask, :]
print("Final matrix shape:", left_active_matrix.shape)

(
    fig_left_active_raster,
    ax_left_active_raster,
    im_left_active_raster,
    left_active_neuron_order,
) = plott.plot_allfish_flat_raster(
    data=left_active_matrix,
    trial_aligned_traces=reference_fish["trial_aligned_traces_raster"],
    stim_order=left_stim_order,
    stimuli_id_map=reference_fish["stimuli_id_map"],
    stimuli_durations=reference_fish["stimuli_durations"],
    stimuli_colors=stimuli_colors,
    stimuli_linestyles=stimuli_linestyles,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    combine_mode=left_combine_mode,
    sort_mode="corravg",
    n_clusters=3,
    random_state=0,
    neuron_order=left_zscore_neuron_order,
    sort_label=None,
    is_binary=True,
    show_mean_trace=True,
    figsize=(12, 8),
    fish_id="all_fish",
    vmax=1,
)
plt.show()


In [ ]:
# Cell 13 - Pooled mean traces and motion-delta distributions
# Restores the original pooled normalized mean traces for seven left-side conditions.
concat_mean_traces = {stim: [] for stim in response_stimulus_ids}
concat_mean_all_neurons = {stim: [] for stim in response_stimulus_ids}
concat_mean_all_neurons_norm = {stim: [] for stim in response_stimulus_ids}

for fid in fish_ids:
    fish = all_fish_data[fid]
    kept_neuron_indices = fish["kept_neuron_indices"]
    for stim in response_stimulus_ids:
        dfof_arr = fish["trial_aligned_traces"][stim]
        dfof_kept = dfof_arr[kept_neuron_indices, :, :]
        mean_trace = np.nanmean(dfof_kept, axis=2)
        concat_mean_traces[stim].append(mean_trace)
        concat_mean_all_neurons[stim].append(np.nanmean(mean_trace, axis=0))

        z_arr = fish["trial_aligned_traces_z_core"][stim]
        mean_trace_norm = np.nanmean(z_arr, axis=2)
        concat_mean_all_neurons_norm[stim].append(np.nanmean(mean_trace_norm, axis=0))

for stim in response_stimulus_ids:
    concat_mean_traces[stim] = np.vstack(concat_mean_traces[stim])
    concat_mean_all_neurons[stim] = np.vstack(concat_mean_all_neurons[stim])
    concat_mean_all_neurons_norm[stim] = np.vstack(concat_mean_all_neurons_norm[stim])

stimuli_wanted = [6, 2, 3, 4, 5, 0, 12]
stimuli_ids_for_selection = response_stimulus_ids
stimuli_names_for_selection = response_stimulus_labels
selected_stimuli_ids = [stimuli_ids_for_selection[pos] for pos in stimuli_wanted]
selected_stimuli_names = [stimuli_names_for_selection[pos] for pos in stimuli_wanted]
mean_trace_selection = {stim_id: concat_mean_all_neurons_norm[stim_id] for stim_id in selected_stimuli_ids}

print("new_stimuli_ids:  ", selected_stimuli_ids)
print("new_stimuli_names:", selected_stimuli_names)
print("shape of first M:", mean_trace_selection[selected_stimuli_ids[0]].shape)

fig_mean_traces, ax_mean_traces, used_colors, out_path = plott.plot_stimulus_means(
    mean_traces=mean_trace_selection,
    stimuli_ids=selected_stimuli_ids,
    stimuli_names=selected_stimuli_names,
    title_prefix="",
    fps_2p=fps_2p,
    t_post_s=t_post_s,
    t_pre_s=t_pre_s,
    stimuli_durations=stimuli_durations,
    plots_path=None,
    prefix=None,
    dpi=600,
    save=False,
    stimuli_colors=stimuli_colors,
    stimuli_linestyles=stimuli_linestyles,
    close_after=False,
    kept_cells=None,
    comment="all_stimuli",
)

fish_id = fish_ids[0]
fish_data = all_fish_data[fish_id]
print("Available stimuli (input_index, stim_id, real_name):")
for idx, (stim_id, stim_name) in enumerate(zip(stimuli_ids_for_selection, stimuli_names_for_selection)):
    print(f"{idx}: id={stim_id}, name={stim_name}")
print("Selected stimuli (input_index, stim_id, real_name):")
for pos, stim_id, stim_name in zip(stimuli_wanted, selected_stimuli_ids, selected_stimuli_names):
    print(f"{pos}: id={stim_id}, name={stim_name}")

kept_neuron_indices = fish_data["kept_neuron_indices"]
trial_aligned_z_kept = {
    stim_id: arr[kept_neuron_indices, :, :]
    for stim_id, arr in fish_data["trial_aligned_traces_z_core"].items()
}
selected_trial_aligned_z_kept = {
    stim_id: trial_aligned_z_kept[stim_id]
    for stim_id in selected_stimuli_ids
}

delta_df = at.compute_motion_delta_integrals(
    selected_trial_aligned_z_kept,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    pre_motion_fixed_s=8.0,
    stimuli_id_map=fish_data["stimuli_id_map"],
    stimuli_durations=fish_data["stimuli_durations"],
    fish_id=fish_id,
)
peak_df = at.compute_motion_delta_peaks(
    selected_trial_aligned_z_kept,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    pre_motion_fixed_s=8.0,
    stimuli_id_map=fish_data["stimuli_id_map"],
    stimuli_durations=fish_data["stimuli_durations"],
    fish_id=fish_id,
)
peak_df = peak_df[peak_df["stimulus"].isin(selected_stimuli_names)].copy()
delta_df = delta_df[delta_df["stimulus"].isin(selected_stimuli_names)].copy()

if delta_df.empty or peak_df.empty:
    raise ValueError("No motion-delta rows were computed for the selected stimuli.")
print("Plotting stimuli:", selected_stimuli_names)

plott.plot_motion_delta_distribution(
    peak_df,
    value_col="delta_peak",
    stimuli=selected_stimuli_names,
    title="Peak delta by selected stimulus",
)
plott.plot_motion_delta_distribution(
    delta_df,
    stimuli=selected_stimuli_names,
    title="Integral delta by selected stimulus",
)


In [ ]:
# Cell 14 - Active-neuron overlap and pooled decision diagnostic
# Restores the original seven-condition left/right Jaccard output and left-side diagnostic plot.
condition_labels = ["Bcontrol", "B1", "B2", "B3", "B4", "FlickL", "Rock"]
side_stimuli = {
    "left": [7, 3, 4, 5, 6, 1, 13],
    "right": [12, 8, 9, 10, 11, 2, 14],
}

active_matrices_overlap = mfa.build_active_neuron_matrices_all_fish(
    all_fish_data=all_fish_data,
    fish_ids=fish_ids,
    stim_order=side_stimuli["left"] + side_stimuli["right"],
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
)

if "plot_neuron_keep_mask" in globals():
    if plot_neuron_keep_mask.shape[0] != response_row_metadata.shape[0]:
        raise ValueError("plot_neuron_keep_mask length does not match response_row_metadata rows.")

    filtered_active_matrices = {}
    for fid in fish_ids:
        fish_rows = response_row_metadata["fish_id"].to_numpy() == fid
        fish_keep_mask = plot_neuron_keep_mask[fish_rows]
        active_matrix = active_matrices_overlap[fid]
        if active_matrix.shape[0] != fish_keep_mask.shape[0]:
            raise ValueError(
                f"Active matrix rows for {fid} do not match response metadata rows: "
                f"{active_matrix.shape[0]} vs {fish_keep_mask.shape[0]}."
            )
        if hasattr(active_matrix, "loc"):
            filtered_active_matrices[fid] = active_matrix.loc[fish_keep_mask].reset_index(drop=True)
        else:
            filtered_active_matrices[fid] = active_matrix[fish_keep_mask, :]
    active_matrices_overlap = filtered_active_matrices
    print(
        "Notebook-level neuron filter for pooled overlap kept "
        f"{int(plot_neuron_keep_mask.sum())} of {plot_neuron_keep_mask.shape[0]} neurons."
    )

overlap_results = mfa.build_active_neuron_overlap_matrices_all_fish(
    active_matrices=active_matrices_overlap,
    side_stimuli=side_stimuli,
    condition_labels=condition_labels,
)
left_overlap_matrix = overlap_results["pooled"]["left"]
right_overlap_matrix = overlap_results["pooled"]["right"]

side_to_plot = "left"
aggregation_to_plot = "mean_per_fish"
matrix_to_plot = overlap_results[aggregation_to_plot][side_to_plot]

display(left_overlap_matrix)
display(right_overlap_matrix)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    matrix_to_plot,
    vmin=0,
    vmax=1,
    square=True,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    cbar_kws={"label": "Jaccard overlap"},
    ax=ax,
)
ax.set_title(f"{side_to_plot.capitalize()} active-neuron overlap")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

# Active-decision strictness diagnostic, matching the original left-side default.
diagnostic_stim_order = side_stimuli["left"]
diagnostic_combine_mode = "mean"
active_matrices_diagnostic = mfa.build_active_neuron_matrices_all_fish(
    all_fish_data=all_fish_data,
    fish_ids=fish_ids,
    stim_order=diagnostic_stim_order,
    fps_2p=fps_2p,
    t_pre_s=t_pre_s,
    tau_s=6.0,
    active_fraction_threshold=0.10,
    min_epoch_s=2.0,
    min_active_reps=2,
    expected_reps=4,
)
active_trace_diagnostic = mfa.build_pooled_active_trace_diagnostic(
    all_fish_data=all_fish_data,
    active_matrices=active_matrices_diagnostic,
    fish_ids=fish_ids,
    stim_order=diagnostic_stim_order,
    combine_mode=diagnostic_combine_mode,
)

if "plot_neuron_keep_mask" in globals():
    if plot_neuron_keep_mask.shape[0] != active_trace_diagnostic["trace_matrix"].shape[0]:
        raise ValueError("plot_neuron_keep_mask length does not match active_trace_diagnostic rows.")
    active_trace_diagnostic = active_trace_diagnostic.copy()
    active_trace_diagnostic["trace_matrix"] = active_trace_diagnostic["trace_matrix"][plot_neuron_keep_mask, :]
    active_trace_diagnostic["decision_matrix"] = active_trace_diagnostic["decision_matrix"][plot_neuron_keep_mask, :]
    active_trace_diagnostic["row_metadata"] = active_trace_diagnostic["row_metadata"].loc[
        plot_neuron_keep_mask
    ].reset_index(drop=True)
    print(
        "Notebook-level neuron filter kept "
        f"{int(plot_neuron_keep_mask.sum())} of {plot_neuron_keep_mask.shape[0]} neurons."
    )

if active_trace_diagnostic["trace_matrix"].shape[0] == 0:
    fig = None
    axes = None
    diagnostic_neuron_order = np.array([], dtype=int)
    print("No neurons passed the current filters; diagnostic plot was not drawn.")
else:
    fig, axes, diagnostic_neuron_order = plott.plot_active_trace_decision_diagnostic(
        active_trace_diagnostic,
        fps_2p=fps_2p,
        stimuli_durations=stimuli_durations,
        t_pre_s=t_pre_s,
        sort_mode="decision_then_mean",
        show_active_count_trace=True,
        active_count_threshold=0.5,
        figsize=(12, 7),
    )
    plt.show()

print("trace_matrix:", active_trace_diagnostic["trace_matrix"].shape)
print("decision_matrix:", active_trace_diagnostic["decision_matrix"].shape)
print("Final neuron_summary_table:", neuron_summary_table.shape)
